# tools

> The toolset is `shalya`. This module is what Ramabana adds to it, and the names it keeps resolving.

In [ ]:
#| default_exp tools

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import sys, tempfile
from contextlib import contextmanager
from pathlib import Path
from fastcore.test import test_eq, test_fail
from ramabana.core import ModelSpec
from ramabana.testing import FakeBackend, MemHost

def names(ts): return {t.__name__ for t in ts}

@contextmanager
def mock_caps(by_id):
    "Answer `spec_caps` from a table, so `draws_itself` can be read without loading a model."
    import ramabana.core as rc
    was = rc._caps
    rc._caps = lambda model_id, runtime: by_id.get(model_id)
    try: yield
    finally: rc._caps = was

The host, the tool factories, the tool-result conventions and the skills registry moved to
[shalya](https://github.com/vedicreader/shalya). Ramabana re-exports every one of them, so
`from ramabana.tools import ...` keeps working and Leela's `sys.modules[__name__] = ramabana.tools`
keeps resolving.

What stays here needs an agent. Delegation needs a `Backend` and a `Run`. `draws_itself` needs a
`ModelSpec`. Neither is something a toolset should know about.

In [ ]:
#| export
import concurrent.futures, functools, json, re, threading, time, uuid
from fastcore.basics import AttrDict, ifnone
from fastcore.foundation import L
from fastcore.parallel import parallel
from shalya.core import (ERR, GIT_READ_TOOLS, GIT_TOOLS, GIT_WRITE_TOOLS, Hit, MAX_API, MAX_FILE,
                         MAX_GREP_HITS, MAX_HITS, MAX_TOOL_CHARS, apply_edits, clip,
                         clip_lines, cmds, diff_text, edits, err, failed, is_write, writes,
                         ACTING_TOOLS, acts, has_effect, summarise, summary,
                         one_line, one_line as _1)
from shalya.host import (Capability, DENY, Host, HostError, LD_CHARS, LocalHost, MAX_VARS, NO_ROOTS,
                         SANDBOX, SECRET, SKIP_DIRS, SKIP_SUFFIXES, denied, implemented, ld_json, _fuse,
                         CodeHost, WebHost, NotebookHost, MemoryHost, WatchHost, SessionHost,
                         ShellHost, ApiHost, GitHost)
from shalya.skills import (EVENTS, EXTRA_MODULES, GROUP, MAX_SKILL_CHARS, Registry, SKILL_DESC_MAX,
                           Skill, discover, ext_dirs, find, load, skill_dirs, skill_index)
from shalya.tools import (API_VENDORS, GROUPS, IMAGE_API, IMAGE_MODEL, IMAGE_SIZES, RESPONSES_API,
                          _post_responses, api_model, api_tools, ask_tools, code_tools, file_tools,
                          image_available, media_dir, memory_tools, mime_for, notebook_tools,
                          readable, save_media, session_tools, shell_tools, skill_tools, watch_tools,
                          web_tools, git_tools)
from shalya.tools import read_only
from shalya.tools import image_tools as _image_tools
from shalya.tools import tools_for as _tools_for
from ramabana.core import AgentError, agent_err, spec_caps
from ramabana.runtime import Run, current_run, run_context

In [ ]:
#| export
from fastcore.docments import frontmatter
from shalya.core import WRITE_TOOLS as _TOOL_WRITES

#: Names Leela imports from `ramabana.tools`. They are shalya's now, under the spellings Leela knows.
_cmds, _edits, _apply_edits, _diff = cmds, edits, apply_edits, diff_text

#: `cart_tools` is an extension rather than a host group, so its two spenders are added by name here.
WRITE_TOOLS = _TOOL_WRITES | {'cart_add', 'cart_remove'}

In [ ]:
#| export
#: shalya's names, re-exported so `from ramabana.tools import *` still finds them.
_all_ = ['frontmatter', 'API_VENDORS', 'Capability', 'DENY', 'ERR', 'EVENTS', 'EXTRA_MODULES', 'GIT_READ_TOOLS', 'GIT_TOOLS', 'GIT_WRITE_TOOLS', 'GROUP', 'GROUPS', 'Hit', 'Host', 'HostError', 'IMAGE_API', 'IMAGE_MODEL', 'IMAGE_SIZES', 'LD_CHARS', 'LocalHost', 'MAX_API', 'MAX_FILE', 'MAX_GREP_HITS', 'MAX_HITS', 'MAX_SKILL_CHARS', 'MAX_TOOL_CHARS', 'MAX_VARS', 'NO_ROOTS', 'RESPONSES_API', 'Registry', 'SANDBOX', 'SECRET', 'SKILL_DESC_MAX', 'SKIP_DIRS', 'SKIP_SUFFIXES', 'Skill', '_apply_edits', '_cmds', '_diff', '_edits', 'api_model', 'api_tools', 'ask_tools', 'apply_edits', 'clip', 'clip_lines', 'cmds', 'code_tools', 'denied', 'diff_text', 'discover', 'edits', 'err', 'ext_dirs', 'failed', 'file_tools', 'find', 'git_tools', 'image_available', 'implemented', 'is_write', 'acts', 'has_effect', 'ACTING_TOOLS', 'summary', 'summarise', 'one_line', 'read_only', 'ld_json', '_fuse', 'CodeHost', 'WebHost', 'NotebookHost', 'MemoryHost', 'WatchHost', 'SessionHost', 'ShellHost', 'ApiHost', 'GitHost', 'load', 'media_dir', 'memory_tools', 'mime_for', 'notebook_tools', 'readable', 'save_media', 'session_tools', 'shell_tools', 'skill_dirs', 'skill_index', 'skill_tools', 'watch_tools', 'web_tools', 'writes']

Every one of those is shalya's object, not a copy. A re-export that drifted into a second
implementation would be worse than no re-export at all. So the identity is asserted here, and
shalya's own notebooks test what these do.

In [ ]:
len(_all_), sorted(_all_)[:8]

In [ ]:
import shalya.core, shalya.host, shalya.skills, shalya.tools
here = sys.modules[__name__] if False else __import__('ramabana.tools', fromlist=['x'])
missing, different = [], []
for n in _all_:
    if not hasattr(here, n): missing.append(n); continue
    src = next((m for m in (shalya.core, shalya.host, shalya.skills, shalya.tools)
                if hasattr(m, n.lstrip('_')) or hasattr(m, n)), None)
    if src is None: continue
    theirs = getattr(src, n, None) or getattr(src, n.lstrip('_'), None)
    if getattr(here, n) is not theirs: different.append(n)
test_eq(missing, [])
test_eq(different, [])
test_eq((_cmds, _edits, _apply_edits, _diff),
        (shalya.core.cmds, shalya.core.edits, shalya.core.apply_edits, shalya.core.diff_text))

`WRITE_TOOLS` is the one name here that is not shalya's. `cart_tools` is an extension rather than a
host group, so its two spenders are added by name.

In [ ]:
sorted(WRITE_TOOLS - shalya.core.WRITE_TOOLS)

In [ ]:
test_eq(WRITE_TOOLS - shalya.core.WRITE_TOOLS, {'cart_add', 'cart_remove'})
assert shalya.core.WRITE_TOOLS < WRITE_TOOLS, 'nothing shalya marks may be dropped here'

A real host over a real folder. Everything below runs against it.

In [ ]:
root = Path(tempfile.mkdtemp()).resolve()/'proj'
(root/'pkg').mkdir(parents=True)
(root/'pkg'/'sizes.py').write_text('def threshold(n):\n    "Half of n."\n    return n // 2\n')
local = LocalHost([root], index=False)
sorted(local.provides)

In [ ]:
test_eq(sorted(local.provides), ['code', 'file', 'git', 'notebook', 'session', 'shell', 'web'])
test_eq(local.can('file'), True)

## A host with nothing behind it

`NullHost` is what the harness runs on in a test: the path boundary, refusing. It declares no
capability group beyond the file one every host has, so `tools_for` gives it the file tools and
nothing else.

In [ ]:
#| export
@implemented
class NullHost(Host):
    "A host with nothing behind it. The reference implementation of 'this host cannot'."

    def __init__(self, roots=()): self._roots = [str(r) for r in roots]

    @property
    def roots(self): return self._roots
    def check(self, path, must_exist=False, reading=False): raise HostError(f'{NO_ROOTS}: {path}')
    def walk(self): return []
    def read(self, path): return None
    def write(self, path, text): raise HostError('this host cannot write')
    def text_at(self, path): return None

## Drawing

Whether the turn's own model can draw is a fact about the model. Shalya takes it as a callable, and
this is where the callable comes from. `Caps.tools` is the answer that matters:
`supported_output_modalities` says no for every chat model, because it describes what one returns
unprompted.

In [ ]:
#| export
def draws_itself(spec):
    "Can `spec`'s own model draw, given the image tool?"
    if spec is None: return False
    c = spec_caps(spec)
    return bool(c is not None and 'image' in getattr(c, 'tools', ()))

def _from_responses(raw):
    "Generated pictures out of a Responses reply, read by the same `rishi` code a turn uses."
    from rishi.remote import gen_media
    return gen_media(raw)

def image_tools(host, mx=MAX_TOOL_CHARS, session='', get_spec=None, on_media=None):
    "shalya's image group, told what this turn's model can do."
    spec = get_spec() if get_spec else None
    return _image_tools(host, mx, session, draws_itself=lambda: draws_itself(spec),
                        from_reply=_from_responses, model_id=getattr(spec, 'model_id', ''),
                        on_media=on_media)

Whether the turn's own model can draw. `supported_output_modalities` says no for every chat model,
because it describes what one returns unprompted. `Caps.tools` is the answer that matters, and it is
what makes a model draw as itself rather than handing the prompt to a different one.

In [ ]:
from types import SimpleNamespace
draws = ModelSpec('drawer', 'remote', 'openai/gpt-5.6-luna')
plain = ModelSpec('plain', 'remote', 'openai/gpt-5.6')
caps = {'openai/gpt-5.6-luna': SimpleNamespace(tools=('image', 'search')),
        'openai/gpt-5.6': SimpleNamespace(tools=('search',))}
with mock_caps(caps): got = draws_itself(draws), draws_itself(plain), draws_itself(None)
got

In [ ]:
with mock_caps(caps):
    test_eq(draws_itself(draws), True)
    test_eq(draws_itself(plain), False)                   # it has tools, and none of them draws
test_eq(draws_itself(None), False)
with mock_caps({}): test_eq(draws_itself(draws), False)   # nothing known about it is not a yes

`image_tools` is shalya's, told what this turn's model can do. Without a key it says so and reaches
nothing, which is what makes this cell runnable.

In [ ]:
gen = image_tools(local, get_spec=lambda: draws)[0]
gen.__name__, image_available()

In [ ]:
test_eq(gen.__name__, 'generate_image')
if not image_available(): assert 'OPENAI_API_KEY' in gen('a kettle')
assert failed(gen('a kettle', size='3x3'))                # an unknown size never reaches the wire
test_eq(len(image_tools(local, get_spec=lambda: None)), 1)

## Assembling the tool list

`tools_for` is shalya's. Ramabana keeps its own signature, because `get_spec` and `on_media` belong
to a turn rather than to a host, and builds the image group before handing it over.

In [ ]:
#| export
def tools_for(host, get_skills=None, extra=(), mx=MAX_TOOL_CHARS, drop=(), get_spec=None, on_media=None):
    """Every tool this host declares it can support, plus whatever extensions registered.

    Groups come from `Host.provides`. `drop` withholds a group the host does support, decided by
    `core.budget_for` and reported by `Agent.budget`.
    """
    # credentialled, never probed: a key is either there or it is not
    image = (image_tools(host, mx, get_spec=get_spec, on_media=on_media)
             if image_available() and 'image' not in set(drop or ()) else None)
    return _tools_for(host, get_skills=get_skills, extra=extra, mx=mx, drop=drop, image=image)

Ramabana's `tools_for` keeps its own signature. `get_spec` and `on_media` belong to a turn rather
than to a host, so the image group is built here and handed to shalya's.

In [ ]:
sorted(t.__name__ for t in tools_for(local))

In [ ]:
ts = tools_for(local)
names = {t.__name__ for t in ts}
assert {'search_code', 'view_file', 'run_python', 'run_shell'} <= names, sorted(names)
assert not (names & {'memory_tree', 'api_load', 'ask_memory'})   # no store, no specs
test_eq('generate_image' in names, image_available())      # credentialled, never probed
test_eq(names - {t.__name__ for t in tools_for(local, drop=['shell'])}, {'run_shell'})
assert 'read_skill' in {t.__name__ for t in tools_for(local, get_skills=lambda: [])}
assert 'read_skill' not in names                           # no skills asked for, so none offered

In [ ]:
h = NullHost(['/proj'])
sorted(h.provides), [t.__name__ for t in tools_for(h)]

In [ ]:
test_eq(sorted(h.provides), ['file'])
test_eq(h.roots, ['/proj'])
test_fail(lambda: h.check('a.py'), contains='no folders are open')

Every method of the boundary refuses, and `tools_for` gives it the file tools alone. This is the
reference implementation of "this host cannot".

In [ ]:
h.walk(), h.read('a.py'), h.text_at('a.py')

In [ ]:
test_eq(h.walk(), [])
test_eq(h.read('a.py'), None)
test_eq(h.text_at('a.py'), None)
test_fail(lambda: h.write('a.py', 'x'), contains='cannot write')
test_fail(lambda: h.check('a.py'), contains='no folders are open')
test_fail(lambda: NullHost().add_root('/tmp'), contains='cannot open another folder')

## Sub-agents

Delegation is a context strategy, not a speed one. A broad question that takes twenty tool calls to answer costs the caller one question and one answer, because the sub-agent's working is discarded with its conversation. A sub-agent gets read-only tools, and cannot delegate further: recursion here is a fan-out tree whose width nobody chose.

In [ ]:
#| export
SUB_MAX_STEPS = 12

SUB_SP = """You are a research sub-agent inside a Python IDE. Another agent has delegated one \
question to you because answering it takes many tool calls and the answer is short.

- Answer exactly the question asked. Nothing else.
- Use your tools as much as you need; nobody is paying attention to how many calls it takes.
- Report what you found, with file paths and line numbers, not what you infer or expect.
- If the answer is that there is nothing, say so plainly. A confident wrong answer is far \
worse than "no matches, and here is what I searched for".
- You cannot edit anything. If the answer implies a change, describe the change and stop.
- `inspect_python` answers questions about the user's live variables without changing them. \
Its default scope is a sandbox that refuses most library calls; pass `scope='overlay'` to \
get the real interpreter. Use it rather than guessing at what is in memory."""


#: Swapped in for the two read-only lines above when a session grants sub-agents writes.
SUB_WRITE_SP = """- You have the delegating agent's write tools as well as its read tools: create and \
edit files, run commands, run Python. Every call is recorded on the session and goes through the \
approval policy the main agent answers to. A refusal comes back with a reason. Read it and change \
the approach.
- Write only what the task asked for. You cannot see the conversation that sent you. Anything \
else you change is a change nobody reviewed.
- Verify with the tool that proves it. Run the test. Read the file back. Report the evidence.
- `run_python` shares the user's kernel namespace. Bind results to NEW names. You cannot rebind or \
delete what the user made."""


def sub_briefing(writes=False):
    "The sub-agent standing instructions, with the read-only sentences swapped out when writes are on."
    if not writes: return SUB_SP
    keep = [ln for ln in SUB_SP.splitlines()
            if not ln.startswith('- You cannot edit anything') and not ln.startswith('- `inspect_python`')]
    return '\n'.join(keep).rstrip() + '\n' + SUB_WRITE_SP


# A sub-agent does not spawn sub-agents: recursion here is a fan-out tree whose width nobody chose.
# Nor does it open standing work: a folder watch outlives the task that opened it, and nobody
# asked for the reviews it would keep producing after the delegation is forgotten.
NO_SUB = frozenset({'delegate_search', 'delegate_parallel', 'delegate_async', 'delegate_status',
                    'delegate_result', 'delegate_cancel',
                    'watch_folder', 'cancel_folder_watch', 'check_folders'})

A sub-agent is briefed as a sub-agent. The read-only prompt is the default, and a session that
grants writes swaps in the paragraph that says what a write costs.

In [ ]:
SUB_MAX_STEPS, len(SUB_SP.split()), len(SUB_WRITE_SP.split())

In [ ]:
assert 'Answer exactly the question asked' in SUB_SP
assert 'cannot edit anything' in SUB_SP                   # the read-only default says so
assert 'approval policy' in SUB_WRITE_SP and 'Verify with the tool that proves it' in SUB_WRITE_SP
assert SUB_MAX_STEPS > 0
# a sub-agent spawns none of these, and collects no background delegation it did not start
test_eq(sorted(NO_SUB), ['cancel_folder_watch', 'check_folders', 'delegate_async', 'delegate_cancel',
                         'delegate_parallel', 'delegate_result', 'delegate_search', 'delegate_status',
                         'watch_folder'])

`read_only` is shalya's. Ramabana passes `block=NO_SUB`, its own delegation tools and the folder
watches that would outlive the task that opened them. `effects=False` withholds the rest of the ways
to act, for a surface that may look and propose but never act: shalya marks those with `acts` and
names them in `ACTING_TOOLS`. What survives can still be pinned, so `read_url` keeps its page and
loses the vault entry it wrote by default.

In [ ]:
[t.__name__ for t in read_only(ts, block=NO_SUB)]

['search_code',
 'grep',
 'ls',
 'similar_code',
 'outline',
 'list_files',
 'view_file',
 'web_search',
 'read_url',
 'research']

In [ ]:
import inspect
test_eq(set(t.__name__ for t in read_only(ts, block=NO_SUB)) & WRITE_TOOLS, set())
shut = set(t.__name__ for t in read_only(ts, effects=False, block=NO_SUB))
test_eq(shut & (WRITE_TOOLS | NO_SUB | ACTING_TOOLS), set())
test_eq('search_code' in shut, True)

kept = []
class Page: text = 'page'
class Keeps(MemHost, WebHost):            # the web group is declared, not duck-typed
    def web_search(self, query, n=20): return []
    def research(self, query): return ''
    def read_url(self, url, remember=True): kept.append(remember); return Page()
fetch = next(t for t in read_only(tools_for(Keeps({'/proj/a.py': 'x = 1\n'}))) if t.__name__ == 'read_url')
fetch(url='https://example.com')
test_eq(kept, [False])                      # the page is kept, the vault entry is not
# shalya swaps in the safe variant rather than pinning the argument, so `remember` never reaches
# the schema the model is given and there is nothing left for it to ask for
test_eq('remember' in inspect.signature(fetch).parameters, False)
sorted(set(t.__name__ for t in ts) & ACTING_TOOLS)

['delegate_parallel', 'delegate_search']

With a budget, the tools themselves stop the loop. Local engines own their internal tool loop. The wrapper is the one hard stop that works on every backend.

In [ ]:
budgeted = read_only(ts, max_calls=1, block=NO_SUB)
search = next(t for t in budgeted if t.__name__ == 'search_code')
search('return'), search('return')

('[memory]\n/proj/a.py:1    def a(): return 1\n  FILE -- use this exact path with view_file/edit_file\n/proj/b.py:1    def b(): return a() + 1\n  FILE -- use this exact path with view_file/edit_file',
 'Sub-agent tool budget exhausted. Stop calling tools and return the best evidence-backed answer now.')

`delegate` runs one question in a throwaway conversation on the same engine, and closes it in a `finally`. A sub-agent whose context leaks back into the session is just a slower way of doing the work inline.

In [ ]:
#| export
def _delegate_result(text):
    "Reject degenerate delegated prose before it can be presented as research."
    text = str(text or '').strip()
    words = re.findall(r"[A-Za-z0-9_+.-]+", text.lower())
    if not text: return 'Delegated inspection failed: the sub-agent returned no answer.'
    if len(words) >= 12 and max(words.count(w) for w in set(words)) > max(8, len(words) // 4):
        return 'Delegated inspection failed: repetitive output was discarded as unreliable.'
    return text

In [ ]:
#| export
def sub_sp(sp=SUB_SP, skills=()):
    "A sub-agent's briefing: its standing instructions, then the bodies of the skills its task named."
    if not skills: return sp
    return sp + '\n\n' + '\n\n'.join(f'## {s.name}\n\n{s.text()}' for s in skills)


def _stopped(run):
    "A stopped delegation answers in text: a dict would reach the model as its own repr."
    return f'The delegated question was stopped ({run.state}) before it answered.'


def bad_json(e, span=120):
    "The fragment a JSON error is about."
    doc, pos = getattr(e, 'doc', None), getattr(e, 'pos', None)
    if not isinstance(doc, str) or not isinstance(pos, int): return ''
    lead = '…' if pos > span else ''
    tail = '…' if pos + span < len(doc) else ''
    return f'\nit stopped here: {lead}{doc[max(0, pos - span):pos + span]}{tail}'


def _model_refused(sub, reply):
    "Whether what came back is the sub-agent's backend reporting its own failure, not an answer."
    problems = getattr(sub, 'problems', None) or []
    return bool(problems) and str(reply or '').strip() == str(problems[-1]).strip()


def delegate(backend, question, tools=(), sp=None, max_steps=SUB_MAX_STEPS, skills=(),
             writes=False,      # hand over WRITE_TOOLS as well
             approve=None,      # the gate those writes answer to, which `spawn` inherits none of
             run=None):         # a pre-registered child run
    "Ask `question` in a throwaway conversation on `backend`'s engine. Returns the answer text."
    sub = None
    run = run or Run(f'run_{uuid.uuid4().hex[:12]}', 'child', str(question), backend.spec.name, current_run())
    if not run.start(): return _stopped(run)
    try:
        # the tool wrappers are the hard stop: native engines own their own tool loop
        kw = {'approve': approve} if approve is not None else {}
        sub = backend.spawn(sp=sub_sp(ifnone(sp, sub_briefing(writes)), skills),
                            tools=read_only(tools, max_calls=max_steps * 4, writes=writes, block=NO_SUB), **kw)
        if hasattr(sub, 'max_steps'): sub.max_steps = max_steps
        if not run.attach(sub): return _stopped(run)
        with run_context(run): reply = sub.send(question, run=run)
        if run.cancelled: return _stopped(run.finish())
        if _model_refused(sub, reply):
            run.finish('failed')
            return err(f'delegation failed after the sub-agent was asked: {reply}')
        run.finish()
        return _delegate_result(reply)
    except Exception as e:
        run.finish('failed')
        return err('delegation failed after the sub-agent was asked' if sub is not None else
                   'delegation failed before the sub-agent was asked, so nothing was sent', e) + bad_json(e)
    finally:
        if sub is not None:
            try: sub.close()
            except Exception: pass

In [ ]:
#| export
def delegate_many(backend, questions, tools=(), sp=None, max_steps=SUB_MAX_STEPS, n_workers=4,
                  skills=(), writes=False, approve=None, parent=None):
    "Ask several questions. Register every child before starting serial or parallel workers."
    qs = L(questions)
    if not qs: return L()
    parent = parent or current_run()
    runs = [Run(f'run_{uuid.uuid4().hex[:12]}', 'child', str(q), backend.spec.name, parent,
                getattr(parent, 'grace', .25)) for q in qs]
    def run(item):
        q, child = item
        if child.cancelled:return _stopped(child)
        return delegate(backend, q, tools, sp, max_steps, skills, writes, approve, child)
    items = list(zip(qs, runs))
    # writing sub-agents stay serial. `Approvals` holds one pending ask, and nobody can review
    # concurrent edits to one workspace
    workers = 1 if writes or getattr(backend.spec, 'local', False) else min(n_workers, len(qs))
    ex = concurrent.futures.ThreadPoolExecutor(max_workers=max(1, workers))
    futures = [ex.submit(run, item) for item in items]
    try:
        while True:
            if all(f.done() for f in futures):break
            if parent is not None and parent.cancelled:
                concurrent.futures.wait(futures, timeout=getattr(parent, 'grace', .25))
                break
            time.sleep(.005)
        out = []
        for child, future in zip(runs, futures):
            if future.done():
                try:out.append(future.result())
                except Exception as e:out.append(err('delegation failed', e) + bad_json(e))
            else:
                child.detach(); out.append(_stopped(child))
        return L(out)
    finally:ex.shutdown(wait=False, cancel_futures=True)

In [ ]:
be = FakeBackend(replies=['the caller never sees this'])
delegate(be, 'which files import fastllm?', tools=ts)

'sub answer'

The spawned conversation is separate, and gone by the time the answer is returned.

In [ ]:
test_eq(len(be.spawned), 1)
be.spawned[0].hist

[{'role': 'user', 'content': 'which files import fastllm?'},
 {'role': 'assistant', 'content': 'sub answer'}]

`delegate_many` keeps the answers in the order the questions were asked. On a local model it runs them one after another on purpose: litert holds one conversation at a time. Fanning out would mean racing for the same engine to find out what happens.

In [ ]:
delegate_many(be, ['what imports fastllm?', 'where is compaction triggered?'], tools=ts)

['sub answer', 'sub answer']

The tools themselves take callables rather than a backend. A model switch mid-session is picked up. The tool the model is holding must not be pinned to whichever backend happened to be current when the list was built.

A delegation that outlives the turn that asked for it. `Background` is the register: a handle
comes back only once the run is in it, so a status call can always find what `delegate_async` just
named. Runs past the ceiling wait as `pending`, and closing the session cancels every one of them.

In [ ]:
#| export
ASYNC_MAX = 4        #: background delegations reaching a model at once; the rest wait as `pending`
ASYNC_KEEP = 50      #: answers held before the oldest is dropped

class Background:
    "Delegations running behind the turn that asked for them, and the answers they leave."
    def __init__(self,
                 mx=ASYNC_MAX,        # how many reach a model at once
                 keep=ASYNC_KEEP):    # answers held before the oldest is dropped
        self.sem = threading.Semaphore(max(1, int(mx)))
        self.keep, self.lock, self.open = max(1, int(keep)), threading.Lock(), True
        self.runs, self.answers, self.seen = {}, {}, set()

    def _live(self, run):
        "Whether `run` should still reach a model. Closing or cancelling stops it, and says so."
        with self.lock: ok = self.open
        if ok and not run.cancelled: return True
        if not run.terminal: run.finish('cancelled')
        return False

    def _answer(self, rid, text):
        with self.lock:
            self.answers[rid], _ = text, self.seen.add(rid)
            for old in list(self.answers)[:-self.keep]: self.answers.pop(old, None)
            # runs outlive their answers by one `keep`, so `status` still names what aged out
            for old in list(self.runs)[:-self.keep * 2]:
                self.runs.pop(old, None); self.seen.discard(old)

    def start(self, fn, run):
        "Register `run`, then work `fn(run)` on a daemon thread. The id names a registered run."
        # registration is the acceptance, and it happens under the lock a close competes for, so a
        # handle is never given out for work this `Background` has already refused to own
        with self.lock:
            if not self.open: raise AgentError('nothing new starts while the session is closing')
            self.runs[run.id] = run
        def work():
            if not self._live(run): return self._answer(run.id, _stopped(run))
            with self.sem:
                if not self._live(run): return self._answer(run.id, _stopped(run))
                try: out = fn(run)
                except Exception as e: out = err('delegation failed', e) + bad_json(e)
            self._answer(run.id, out)
            # after the answer, so a reader never finds a terminal run whose answer is not there yet
            if not run.terminal: run.finish()
        threading.Thread(target=work, daemon=True, name=f'ramabana-bg-{run.id}').start()
        return run.id

    def status(self, run_id=''):
        "Rows for every registered run, or for one. A miss is text saying so, not an empty list."
        with self.lock: runs = dict(self.runs)
        if not run_id: return [r.dict() for r in runs.values()]
        r = runs.get(str(run_id))
        return [r.dict()] if r is not None else f'no delegation named {run_id!r} was started here'

    def result(self, run_id):
        "The answer `run_id` left, or what it is still doing."
        with self.lock:
            rid = str(run_id)
            run, ans, held = self.runs.get(rid), self.answers.get(rid), rid in self.seen
        if run is None: return f'no delegation named {run_id!r} was started here'
        if ans is not None: return ans
        # only an answer this register actually held can have aged out. A terminal run with none
        # is one whose worker has not written it yet, and saying "gone" would end the asking
        if held: return f'{run_id} finished ({run.state}) and its answer is no longer held'
        return f'{run_id} is {run.state}; ask again later'

    def cancel(self, run_id):
        "Stop one run. Its answer becomes the stopped notice the next `result` reads."
        with self.lock: run = self.runs.get(str(run_id))
        if run is None: return f'no delegation named {run_id!r} was started here'
        if run.terminal: return f'{run_id} had already finished ({run.state})'
        run.request_cancel()
        return f'{run_id} is stopping'

    def close(self):
        "Refuse new work and cancel what is running. Queued runs stop rather than reaching a model."
        with self.lock: self.open, runs = False, list(self.runs.values())
        for r in runs: r.request_cancel()
        return len(runs)


In [ ]:
bg = Background(mx=1)          # one slot, so the second has to wait
gate, seen = threading.Event(), []

def slow(run):
    run.start()                # what `delegate` does first, and what makes `running` mean working
    seen.append(run.id)
    gate.wait(5)
    return f'answered {run.question}'

a = bg.start(slow, Run('run_a', 'child', 'first'))
b = bg.start(slow, Run('run_b', 'child', 'second'))
time.sleep(.2)
[(r['id'], r['state']) for r in bg.status()]

In [ ]:
# both are registered before `start` returned, which is what makes a handle usable at once
test_eq({r['id'] for r in bg.status()}, {'run_a', 'run_b'})
test_eq(bg.status('run_a')[0]['state'], 'running')
test_eq(bg.status('run_b')[0]['state'], 'pending')   # the ceiling holds: one queued, not running
test_eq(len(seen), 1)                                # ...and it never reached the callback

assert 'is running' in bg.result('run_a')
assert bg.result('nope').startswith('no delegation named')
assert bg.status('nope').startswith('no delegation named')

gate.set()
for _ in range(200):
    if bg.result('run_b').startswith('answered'): break
    time.sleep(.02)
test_eq(bg.result('run_a'), 'answered first')
test_eq(bg.result('run_b'), 'answered second')
test_eq(len(seen), 2)

In [ ]:
# a run cancelled while it waits stops rather than reaching the callback, and says so
held = Background(mx=1)
block, ran = threading.Event(), []
held.start(lambda r: (r.start(), block.wait(5), ran.append(r.id))[2], Run('run_1', 'child', 'holds the slot'))
held.start(lambda r: ran.append(r.id), Run('run_2', 'child', 'never runs'))
time.sleep(.2)
test_eq(held.cancel('run_2'), 'run_2 is stopping')
block.set()
for _ in range(200):
    if 'stopped' in held.result('run_2'): break
    time.sleep(.02)
assert 'stopped (cancelled)' in held.result('run_2'), held.result('run_2')
test_eq(ran, ['run_1'])                       # the cancelled one emitted nothing
assert held.cancel('run_2').startswith('run_2 had already finished')

In [ ]:
# closing refuses new work and cancels what is registered
shut = Background()
shut.start(lambda r: 'done', Run('run_x', 'child', 'q'))
test_eq(shut.close(), 1)
test_fail(lambda: shut.start(lambda r: 'nope', Run('run_y', 'child', 'q')),
          contains='while the session is closing')
test_eq(shut.status('run_y'), "no delegation named 'run_y' was started here")

# a callback that raises answers with the error rather than losing the run
boom = Background()
boom.start(lambda r: 1/0, Run('run_z', 'child', 'q'))
for _ in range(200):
    if ERR in boom.result('run_z'): break
    time.sleep(.02)
assert 'ZeroDivisionError' in boom.result('run_z'), boom.result('run_z')

# only `keep` answers are held, and a run whose answer aged out says so rather than looking live
small = Background(keep=1)
for n in ('run_p', 'run_q'): small.start(lambda r: f'ans {r.id}', Run(n, 'child', 'q'))
for _ in range(200):
    if small.result('run_q') == 'ans run_q': break
    time.sleep(.02)
assert 'no longer held' in small.result('run_p'), small.result('run_p')

In [ ]:
#| export
def named_skills(get_skills, names):
    "The skills a delegated task named, and a note about any name that matched nothing."
    if not names or get_skills is None: return [], ''
    every = list(get_skills() or [])
    got, missing = [], []
    for n in [x for x in str(names).replace(',', ' ').split() if x]:
        s = find(every, n)
        got.append(s) if s is not None else missing.append(n)
    if not missing: return got, ''
    return got, (f"\n\n[no skill named {', '.join(missing)}; this repository has "
                 f"{', '.join(s.name for s in every) or 'none'}]")


def subagent_tools(get_backend, get_tools, get_skills=None, get_cloud_backend=None,
                   get_writes=None,     # the session's sub-agent write toggle, read per call
                   get_approve=None,    # the gate those writes answer to
                   background=None):    # the register async delegations live in; one is made if None
    """The `delegate` tool, bound to whatever backend routing says sub-agents run on.

    Every argument is a callable. A model switch mid-session is picked up. `get_tools` is the
    sub-agent model's tool list, not the turn model's.
    """

    bg = ifnone(background, Background())

    def _writes(): return bool(get_writes()) if get_writes is not None else False
    def _approve(): return get_approve() if (get_approve is not None and _writes()) else None

    @summary(lambda a: f'Delegate: {_1(a.get("question"), 120)}')
    def delegate_search(question: str, skills: str = '') -> str:
        """Hand a broad search question to a sub-agent and get back only its conclusion.

        Use this when answering would take many `search_code` / `view_file` / `read_url` /
        `inspect_python` calls whose results you do not need to keep. "where else do we
        do X", "which files import Y", "what shape is everything in this namespace". Its
        working is discarded. The cost to your context is one question and one answer.

        The sub-agent has your read-only tools. Whether it also has your write tools is the
        session's setting rather than yours. With sub-agent writes on it can edit, run commands
        and run Python under the approval policy you answer to. The task you send may then ask
        for a change. With them off it can only report.

        `skills` names skills from your skill index, comma separated, whose text the sub-agent
        should start with: name the one or two its task actually needs. You hold the index and
        it does not. This is the only way it gets a skill without spending a step reading
        one. Leave it empty when the task needs no particular skill.

        Ask one self-contained question. The sub-agent cannot see this conversation.
        """
        b = get_backend()
        if b is None: return 'no model is available to delegate to'
        sk, note = named_skills(get_skills, skills)
        return clip(delegate(b, question, get_tools(), skills=sk, writes=_writes(),
                             approve=_approve()), MAX_TOOL_CHARS) + note

    @summary(lambda a: f'Delegate in parallel: {_1(a.get("questions"), 110)}')
    def delegate_parallel(questions: str, skills: str = '', cloud_model: str = '') -> str:
        """Hand several independent questions to sub-agents at once, and get back every answer.

        `questions` is a JSON array of strings, e.g.
          ["which files import fastllm?", "where is compaction triggered?", "what is df's shape?"]

        Use it when you have two or more questions that do not depend on each other. They
        run concurrently, each in its own throwaway conversation with your read-only tools. Three questions cost you three short answers rather than the sixty tool results
        it would take to answer them yourself. With sub-agent writes on they run one after
        another instead, because their approvals share one queue.

        `skills` names skills from your skill index, comma separated, given to every one of
        them. Use it when the questions share a subject. When they do not, ask them in separate
        `delegate_search` calls so each gets only what its own task needs.

        `cloud_model` optionally selects one configured remote model for this fan-out. It does not
        change the session's turn or default sub-agent model. Every question must be self-contained:
        a sub-agent cannot see this conversation or the other questions.
        """
        b = get_cloud_backend(cloud_model) if cloud_model and get_cloud_backend is not None else get_backend()
        if b is None: return f"no model is available to delegate to{f' ({cloud_model})' if cloud_model else ''}"
        try:
            qs = json.loads(questions) if isinstance(questions, str) else list(questions)
            if not isinstance(qs, list) or not all(isinstance(q, str) for q in qs):
                raise ValueError('expected a JSON array of strings')
        except Exception as e:
            return err('could not parse questions', e)
        if not qs: return 'no questions given'
        sk, note = named_skills(get_skills, skills)
        answers = delegate_many(b, qs, get_tools(), skills=sk, writes=_writes(), approve=_approve())
        return clip('\n\n'.join(f'### {q}\n{a}' for q, a in zip(qs, answers)), MAX_TOOL_CHARS * 2) + note

    @summary(lambda a: f'Delegate in the background: {_1(a.get("question"), 110)}')
    def delegate_async(question: str, skills: str = '', writes: bool = False) -> str:
        """Start a sub-agent on `question` and come back for the answer later. Returns a run id.

        Use this when the work takes long enough that waiting for it wastes the turn: a
        survey of a large tree, a refactor across many files, anything you would otherwise
        sit through. The turn you are in can end. A later turn collects the answer.

        Read the id back with `delegate_result`. `delegate_status` says whether it has
        finished, and `delegate_cancel` stops it. An answer is kept until the session ends.

        `writes` is per call and off by default. A background sub-agent is running when
        nobody is watching, so its write tools are withheld unless you ask for them here,
        whatever the session's own sub-agent setting says. Ask for them only when the task
        you are sending is a change rather than a question.

        `skills` names skills from your skill index, comma separated, the same as
        `delegate_search`. Ask one self-contained question: a sub-agent cannot see this
        conversation.
        """
        b = get_backend()
        if b is None: return 'no model is available to delegate to'
        sk, note = named_skills(get_skills, skills)
        w = bool(writes) and _writes()
        # no parent: a background run outlives the turn that started it, and a turn whose child is
        # still live never goes idle. `Background.close` is what stops these, not the turn ending
        child = Run(f'run_{uuid.uuid4().hex[:12]}', 'background', str(question), b.spec.name)
        try: rid = bg.start(lambda r: delegate(b, question, get_tools(), skills=sk, writes=w,
                                               approve=_approve() if w else None, run=r), child)
        except Exception as e: return err('could not start the delegation', e)
        asked = 'with write tools' if w else 'read-only'
        return f'started {rid} ({asked}). Collect it with delegate_result({rid!r}).' + note

    @summary(lambda a: f'Check delegation {a["run_id"]}' if a.get('run_id') else 'Check the background delegations')
    def delegate_status(run_id: str = '') -> str:
        """What a background delegation is doing. No `run_id` lists every one this session started.

        A run reads `pending` while it waits for a free slot, `running` once it has one, and
        `completed`, `cancelled` or `failed` when there is an answer to collect.
        """
        rows = bg.status(run_id)
        if isinstance(rows, str): return rows
        if not rows: return 'nothing has been delegated in the background'
        return clip('\n'.join(f"{r['id']}  {r['state']:10} {r['question'][:80]}" for r in rows), MAX_TOOL_CHARS)

    @summary(lambda a: f'Collect delegation {a.get("run_id","?")}')
    def delegate_result(run_id: str) -> str:
        "The answer a background delegation left, or what it is still doing."
        return clip(bg.result(run_id), MAX_TOOL_CHARS)

    @summary(lambda a: f'Cancel delegation {a.get("run_id","?")}')
    def delegate_cancel(run_id: str) -> str:
        "Stop a background delegation. What it had already done is not undone."
        return bg.cancel(run_id)

    return [delegate_search, delegate_parallel, delegate_async, delegate_status,
            delegate_result, delegate_cancel]

`sub_sp` is the briefing a delegated question is asked under. `named_skills` resolves which skills
go into it, and returns a note for any name that matched nothing rather than failing.

In [ ]:
skills = [Skill('house-style', 'md', 'How code is written here.', _text='Dense lines.'),
          Skill('deploy', 'md', 'How this ships.', _text='Push the tag.')]
named_skills(lambda: skills, 'house-style'), named_skills(lambda: skills, 'nope')

In [ ]:
got, note = named_skills(lambda: skills, 'house-style')
test_eq([s.name for s in got], ['house-style'])
test_eq(note, '')
test_eq([s.name for s in named_skills(lambda: skills, 'house-style,deploy')[0]],
        ['house-style', 'deploy'])
test_eq(named_skills(lambda: skills, ''), ([], ''))
# a name that matched nothing is a note rather than a failure, and it lists what there is
missed, note = named_skills(lambda: skills, 'nope')
test_eq(missed, [])
assert 'no skill named nope' in note and 'house-style, deploy' in note

briefed = sub_sp(skills=got)
assert 'Dense lines.' in briefed and briefed.startswith(SUB_SP)   # the body, not the name
assert 'Push the tag.' not in briefed
test_eq(sub_sp(), SUB_SP)

In [ ]:
subs = {t.__name__: t for t in subagent_tools(lambda: be, lambda: ts)}
delegate_search, delegate_parallel = subs['delegate_search'], subs['delegate_parallel']
list(subs)

['delegate_search', 'delegate_parallel']

With no model available it says so, rather than raising into the turn.

In [ ]:
# with no model behind them, every tool that would ask one says so rather than raising
nomodel = {t.__name__: t for t in subagent_tools(lambda: None, lambda: ts)}
test_eq(nomodel['delegate_search']('anything'), 'no model is available to delegate to')
test_eq(nomodel['delegate_async']('anything'), 'no model is available to delegate to')
delegate_parallel('["what imports fastllm?"]')

'### what imports fastllm?\nsub answer'

The async four sit on the same `Background`. A handle comes back at once, the answer later.

In [ ]:
bg_be = FakeBackend()
bgs = {t.__name__: t for t in subagent_tools(lambda: bg_be, lambda: ts)}

started = bgs['delegate_async']('survey the tree')
rid = started.split()[1]
assert started.startswith('started run_') and 'read-only' in started, started

for _ in range(300):
    if bgs['delegate_status'](rid).split()[1] == 'completed': break
    time.sleep(.02)
test_eq(bgs['delegate_result'](rid), 'sub answer')      # what `FakeBackend.spawn` replies
test_eq(len(bg_be.spawned), 1)                          # ...and it really did spawn one
assert rid in bgs['delegate_status']()
assert bgs['delegate_result']('run_nope').startswith('no delegation named')
assert bgs['delegate_cancel'](rid).startswith(f'{rid} had already finished')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

Parallel delegation registers every child before work starts. Cancelling the parent stops the running child and prevents queued children from spawning.


In [ ]:
class _Sub:
    max_steps = 0
    def __init__(self, owner): self.owner, self.release = owner, threading.Event()
    def send(self, question, run=None):
        self.owner.started.append(question); self.release.wait(); return 'late'
    def cancel(self): self.owner.cancelled += 1; self.release.set(); return True
    def close(self): pass

class _ParentBackend:
    def __init__(self):
        self.spec = AttrDict(name='fake-child', local=False)
        self.started, self.spawned, self.cancelled = [], 0, 0
    def spawn(self, **kw): self.spawned += 1; return _Sub(self)

backend, parent, box = _ParentBackend(), Run('run_parent', grace=.03), []
parent.start()
t = threading.Thread(target=lambda: box.extend(delegate_many(backend, ['a', 'b', 'c'], n_workers=1, parent=parent)), daemon=True)
t.start()
while not backend.started: time.sleep(.001)
test_eq(len(parent.children), 3)
parent.cancel(); t.join(.2)
test_eq((backend.spawned, backend.cancelled), (1, 1))
test_eq(len(box), 3)
# either shape says cancelled, and which one comes back is a matter of whether the worker finished
# inside the join: a future still in flight is detached and reported as its run, one that returned
# carries `delegate`'s own cancellation message. Asserting only the first made this a coin flip
assert all((isinstance(x, dict) and x['state'] in ('cancelled', 'detached'))
           or (isinstance(x, str) and 'cancelled' in x) for x in box)
